# *Data Loading*

In [1]:
import pandas as pd
import requests
from datetime import timedelta

df = pd.read_csv("../data/cleaned_dataset.csv")
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delayed,...,delay_hours_recon,delayed_flag_recon,order_date_recon,order_ts_recon,delivery_ts_recon,expected_ts_recon,hour,order_hour,order_day,is_weekend
0,250.99,amazon logistics,automobile parts,ev bike,standard,west,clear,235.6,48.07,no,...,-43.467511,0,21-10-2024,2024-10-21 13:00:00,2024-10-21 22:19:56.959673890,2024-10-23 17:48:00,13,13,0,0
1,250.99,amazon logistics,clothing,bike,express,central,stormy,81.8,45.51,yes,...,-3.870065,0,02-01-2024,2024-01-02 12:00:00,2024-01-02 16:07:47.764281093,2024-01-02 20:00:00,12,12,1,0
2,250.99,amazon logistics,clothing,van,same day,north,clear,282.9,31.33,yes,...,-18.972602,0,31-05-2024,2024-05-31 11:00:00,2024-05-31 18:25:38.631406710,2024-06-01 13:24:00,11,11,4,0
3,250.99,amazon logistics,cosmetics,ev bike,two day,central,hot,88.6,8.67,no,...,-44.002989,0,03-01-2024,2024-01-03 17:00:00,2024-01-03 20:59:49.240153885,2024-01-05 17:00:00,17,17,2,0
4,250.99,amazon logistics,cosmetics,ev van,two day,east,rainy,204.2,8.09,no,...,-45.866649,0,19-03-2024,2024-03-19 13:00:00,2024-03-19 19:56:00.063604276,2024-03-21 17:48:00,13,13,1,0


In [2]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   delivery_id                25000 non-null  float64
 1   delivery_partner           25000 non-null  object 
 2   package_type               25000 non-null  object 
 3   vehicle_type               25000 non-null  object 
 4   delivery_mode              25000 non-null  object 
 5   region                     25000 non-null  object 
 6   weather_condition          25000 non-null  object 
 7   distance_km                25000 non-null  float64
 8   package_weight_kg          25000 non-null  float64
 9   delayed                    25000 non-null  object 
 10  delivery_status            25000 non-null  object 
 11  delivery_rating            25000 non-null  int64  
 12  delivery_cost              25000 non-null  float64
 13  expected_time_hours_recon  25000 non-null  flo

,delivery_id,distance_km,package_weight_kg,delivery_rating,delivery_cost,expected_time_hours_recon,speed_kmph_recon,weather_mult_recon,delivery_time_hours_recon,partner_mult_recon,delay_hours_recon,delayed_flag_recon,hour,order_hour,order_day,is_weekend
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.0000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,12500.500000,150.390436,25.145898,3.666000,864.944579,33.053344,39.9762,1.100310,5.360406,0.999994,-27.692938,0.029400,12.515400,12.515400,2.987960,0.282400
std,7212.732314,86.409745,14.368663,1.149964,435.712593,17.592341,6.4653,0.064465,2.631007,0.003666,17.636803,0.168928,2.751564,2.751564,1.996551,0.450176
min,250.990000,3.600000,0.670000,1.000000,95.667400,8.000000,30.0000,1.000000,0.640518,0.993437,-48.086207,0.000000,8.000000,8.000000,0.000000,0.000000
25%,6250.750000,75.900000,12.680000,3.000000,490.800000,24.000000,35.0000,1.050000,3.180450,0.998812,-44.388864,0.000000,10.000000,10.000000,1.000000,0.000000
50%,12500.500000,151.000000,25.145000,4.000000,867.535000,26.400000,40.0000,1.100000,5.227587,1.000945,-23.192956,0.000000,12.000000,12.000000,3.000000,0.000000
75%,18750.250000,224.900000,37.660000,5.000000,1237.910000,48.000000,45.0000,1.150000,7.307020,1.002187,-14.075366,0.000000,15.000000,15.000000,5.000000,1.000000
max,24750.010000,297.100000,49.520000,5.000000,1632.720600,52.800000,50.0000,1.200000,13.682192,1.005584,4.860295,1.000000,18.000000,18.000000,6.000000,1.000000


In [3]:
df["order_ts_recon"] = pd.to_datetime(df["order_ts_recon"])
df["expected_ts_recon"] = pd.to_datetime(df["expected_ts_recon"])

df["order_date"] = df["order_ts_recon"].dt.date
df["expected_date"] = df["expected_ts_recon"].dt.date

In [5]:
years = df["order_ts_recon"].dt.year.unique()

print("Years in dataset:", years)

Years in dataset: [2024]


# *Holdiay API*

In [6]:
API_KEY = "JSE0dBGKppjow8bkbfDp10eWORYbxYhe"

all_holidays = []

for year in years:

    url = f"https://calendarific.com/api/v2/holidays?api_key={API_KEY}&country=IN&year={year}"

    response = requests.get(url)
    data = response.json()

    for h in data["response"]["holidays"]:

        all_holidays.append({
            "date": h["date"]["iso"][:10],
            "holiday_name": h["name"]
        })

In [8]:
holiday_df = pd.DataFrame(all_holidays)

holiday_df["date"] = pd.to_datetime(holiday_df["date"]).dt.date

holiday_df.head()

,date,holiday_name
0,2024-01-01,New Year's Day
1,2024-01-13,Lohri
2,2024-01-14,Makar Sankranti
3,2024-01-15,Pongal
4,2024-01-17,Guru Govind Singh Jayanti


In [9]:
holiday_df.to_csv("../data/india_holidays_cached.csv", index=False)

In [11]:
holiday_dict = dict(zip(holiday_df["date"], holiday_df["holiday_name"]))

In [12]:
def get_transit_holidays(start, end):

    holidays = []
    current = start

    while current <= end:

        if current in holiday_dict:
            holidays.append(holiday_dict[current])

        current += timedelta(days=1)

    return holidays

In [14]:
df["transit_holidays"] = df.apply(
    lambda x: get_transit_holidays(x["order_date"], x["expected_date"]),
    axis=1
)

In [15]:
df["holiday_count_transit"] = df["transit_holidays"].apply(len)

df["holiday_names_transit"] = df["transit_holidays"].apply(
    lambda x: ", ".join(x) if len(x) > 0 else "None"
)

In [17]:
def count_weekends(start, end):

    count = 0
    current = start

    while current <= end:

        if current.weekday() >= 5:
            count += 1

        current += timedelta(days=1)

    return count

df["weekend_count_transit"] = df.apply(
    lambda x: count_weekends(x["order_date"], x["expected_date"]),
    axis=1
)

In [18]:
df["holiday_or_weekend_transit_flag"] = (
    (df["holiday_count_transit"] > 0) |
    (df["weekend_count_transit"] > 0)
).astype(int)

In [20]:
def holiday_proximity(order_date):

    min_gap = 999

    for h_date in holiday_dict.keys():

        gap = abs((h_date - order_date).days)

        if gap < min_gap:
            min_gap = gap

    return min_gap

df["holiday_proximity_feature"] = df["order_date"].apply(holiday_proximity)

In [21]:
df[[
    "holiday_count_transit",
    "holiday_names_transit",
    "weekend_count_transit",
    "holiday_or_weekend_transit_flag",
    "holiday_proximity_feature"
]].head()

,holiday_count_transit,holiday_names_transit,weekend_count_transit,holiday_or_weekend_transit_flag,holiday_proximity_feature
0,0,None,0,0,1
1,0,None,0,0,1
2,0,None,1,1,8
3,0,None,0,0,2
4,1,March Equinox,0,1,1


In [22]:
df.to_csv("../data/dataset_with_holidays.csv", index=False)